## Action Responder Prompt

This notebook hopes to:

- Gather trace data for action-responder agent
- Structure and save as mlflow dataset
- Utilize judges to evaluate current agent
- Try with better model to confirm judges are working
- Run GEPA optimzation to improve

In [1]:
import mlflow
from mlflow.tracking import MlflowClient

from summit_sim.settings import settings

mlflow.tracing.enable_notebook_display()

client = MlflowClient()
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
experiment = client.get_experiment_by_name(settings.mlflow_experiment_name)


# Get all traces from last 7 days (adjust as needed)
filter_string = """
tags.graph_type = 'simulation' AND
tags.agent_name = 'action-responder'
"""
traces = client.search_traces(
    locations=[experiment.experiment_id],
    filter_string=filter_string,
    max_results=500,
)

2026-03-29 21:56:41 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:56:41 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:56:41 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:56:41 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:56:41 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:56:41 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:56:41 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlfl

In [2]:
spans = [
    span
    for trace in traces
    for span in client.get_trace(trace.info.trace_id).data.spans
    if span.name == "action_response_agent"
]

dataset = [
    {"inputs": span.inputs["input_data"], "outputs": span.outputs} for span in spans
]

print(f"Found {len(dataset)} action_response_agent spans")

spans

Found 24 action_response_agent spans


[Span(name='action_response_agent', trace_id='tr-affe256ef8a85223f22a7b2f6f8953f9', span_id='dbbb9f7cd9a59114', parent_id='9029006bcb84b67a'),
 Span(name='action_response_agent', trace_id='tr-02676ba69b9f82ffdc08ac94d18f06e2', span_id='2189d23417dddc2b', parent_id='0f9bcffcde27ea61'),
 Span(name='action_response_agent', trace_id='tr-141e739e58049c1a79786b59ee8fdd03', span_id='8b4960e90c0953f5', parent_id='1c49ba6e68b0f081'),
 Span(name='action_response_agent', trace_id='tr-5edc75e07690dd965b51d8b35fb26a86', span_id='1e7a3d0bf0f26664', parent_id='4453717196ef353f'),
 Span(name='action_response_agent', trace_id='tr-2ea58049cc996b1cccb1815df46cec62', span_id='72923819e572101d', parent_id='3c7c52bc6d95e7d2'),
 Span(name='action_response_agent', trace_id='tr-fa63eaabbc1080b664d1dc4aef22e720', span_id='9cef0743cdbb6ffa', parent_id='77a29bef3bd1938d'),
 Span(name='action_response_agent', trace_id='tr-6979387d5e89563634062b84bb2f547c', span_id='81d6190fd3b6bdad', parent_id='11df237beb6fd948'),

[Trace(trace_id=tr-affe256ef8a85223f22a7b2f6f8953f9), Trace(trace_id=tr-02676ba69b9f82ffdc08ac94d18f06e2), Trace(trace_id=tr-141e739e58049c1a79786b59ee8fdd03), Trace(trace_id=tr-5edc75e07690dd965b51d8b35fb26a86), Trace(trace_id=tr-2ea58049cc996b1cccb1815df46cec62), Trace(trace_id=tr-fa63eaabbc1080b664d1dc4aef22e720), Trace(trace_id=tr-6979387d5e89563634062b84bb2f547c), Trace(trace_id=tr-35573d4c1b256e7ca341b17d06b2f062), Trace(trace_id=tr-79f172b56f8eddf75effd7bc01b520e1), Trace(trace_id=tr-050dbc90371a14aa2798144552bef0ac)]

In [24]:
from mlflow.genai.judges import make_judge

JUDGE_MODEL_ENDPOINT = "openrouter:/deepseek/deepseek-v3.2"

COMPLETION_JUDGE_INSTRUCTIONS = """\
You are evaluating the scoring accuracy and pedagogical quality of an
AI-generated response in a wilderness first responder (WFR) training simulation.

=== PATIENT ASSESSMENT SYSTEM (PAS) - GUIDELINES ===

The PAS follows this general order, but students may bundle steps efficiently or adapt based on the situation:

1. SCENE SIZE-UP: 0-.2 points
   - Check for hazards (environmental dangers, unstable terrain, etc.)
   - Identify mechanism of injury (MOI)
   - Count patients and assess available resources

2. PRIMARY ASSESSMENT (ABCDE): 0-.2 points
   - A: Airway assessment
   - B: Breathing evaluation
   - C: Circulation (pulse, bleeding, shock signs)
   - D: Disability (mental status, AVPU)
   - E: Exposure/Environment (clothing, temperature, elements)

3. SECONDARY ASSESSMENT: 0-.2 points
   - Vital signs (HR, RR, BP, SCTM, temperature)
   - Head-to-Toe exam (systematic physical check)
   - SAMPLE history (Signs/Symptoms, Allergies, Medications, Past history, Last intake, Events leading to injury)

4. TREATMENT: 0-.2 points
   - Address immediate life threats
   - Immobilize injuries
   - Wound care
   - Pain management

5. EVACUATION PLAN: 0-.2 points
   - Stay vs. Go decision
   - Resource planning
   - Timeline establishment

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


Evaluate against these criteria:
1. Does completion_score align with the WFR rubric based on cumulative actions?
2. Is the score increase from previous turn reasonable (<=0.2 unless explicit bundling)?
3. Does the score always build on itself? Does completion_score never decrease across turns?
4. If there was no action from the student, the completion_score should not increase from it's previous turn.
    If it did without action, automatically provide a zero score for the agent (THIS SUPERSEDES ANY PREVIOUS ASSESSMENT).

Provide a score of 0-1 on how well the AI Agent performs
"""


completion_judge = make_judge(
    name="completion-judge",
    instructions=COMPLETION_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [25]:
QUALITY_JUDGE_INSTRUCTIONS = """\
You are evaluating the structure and quality of an AI-generated response \
in a wilderness first responder training simulation.

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


Evaluate against these criteria:
1. Does narrative_text end with an open question? Does feedback contain NO questions?
2. Is the feedback personalized to the students response and focus only on history, not future?
3. Is the feedback encouraging and constructive without harsh corrections and focused
    on feedback alone?
4. Is feedback and narrative length between 2-4 sentences?
5. Does the narrative not overtly share hidden information, only revealing
    if the student's action constitutes it? If any extra information was injected in the narrative
    or feedback, automatically provide a zero score for the agent (THIS SUPERSEDES ANY PREVIOUS ASSESSMENT).

Provide a score of 0-1 on how well the AI Agent performs
"""


quality_judge = make_judge(
    name="quality-judge",
    instructions=QUALITY_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [26]:
MEDICAL_JUDGE_INSTRUCTIONS = """\
You are evaluating the medical accuracy of an AI-generated response in a \
wilderness first responder training simulation.

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


TIP: Reference hidden_truth and hidden_state to determine if treatment was premature.

Evaluate against these criteria:
1. Is was_correct accurate?
 - was_correct should be FALSE if student performed treatment \
    (splint, bandage, medication, move patient) without assessment
 - was_correct should be TRUE for assessment actions (vitals, exam, SAMPLE history)
2. Does was_correct accurately gage the quality of the student's action?
3. Is there anything in the AI response that is not medically accurate? If so,
    automatically provide a zero score for the agent (THIS SUPERSEDES ANY PREVIOUS ASSESSMENT).

Provide a score of 0-1 on how well the AI Agent performs
"""


medical_judge = make_judge(
    name="medical-judge",
    instructions=MEDICAL_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [ ]:
scorers = [completion_judge, quality_judge, medical_judge]

# results = mlflow.genai.evaluate(
#     data=dataset,
#     scorers=scorers,
# )

Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?] 

22:40:57 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 22:40:57 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
22:40:57 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 22:40:57 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
22:40:57 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 22:40:57 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
22:40:57 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 22:40:57 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
22:40:57 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2

In [ ]:
from summit_sim.agents.action_responder import ActionRequest, action_response_agent


async def optimize_wrapper(**kwargs) -> dict:  # noqa: ANN003
    """Parse dict into pydantic request to pass to agent."""
    # 1. Pack MLflow's individual kwargs into your Pydantic model
    request_data = ActionRequest(**kwargs)

    # 2. Await your traced LangGraph agent
    response = await action_response_agent(input_data=request_data)

    # 3. Dump the Pydantic response to a dictionary for MLflow compatibility
    return response.model_dump()

In [ ]:
import os

from mlflow.genai.optimize.optimizers import GepaPromptOptimizer

os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"] = "True"

result = mlflow.genai.optimize_prompts(
    predict_fn=optimize_wrapper,
    train_data=dataset,
    prompt_uris=[
        "prompts:/action-responder-system@latest",
        "prompts:/action-responder-user@latest",
    ],
    optimizer=GepaPromptOptimizer(
        reflection_model=JUDGE_MODEL_ENDPOINT,
        display_progress_bar=True,
    ),
    scorers=scorers,
)

/home/bhamm/repos/summit-sim/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


2026-03-29 23:15:28 [INFO] summit_sim.agents.action_responder: Processing student action: turn=5/5, action_length=3
2026-03-29 23:15:28 [INFO] summit_sim.agents.action_responder: Processing student action: turn=4/5, action_length=3
2026-03-29 23:15:28 [INFO] summit_sim.agents.action_responder: Processing student action: turn=3/5, action_length=3
2026-03-29 23:15:28 [INFO] summit_sim.agents.action_responder: Processing student action: turn=2/5, action_length=3
2026-03-29 23:15:28 [INFO] summit_sim.agents.action_responder: Processing student action: turn=1/5, action_length=3
2026-03-29 23:15:28 [INFO] summit_sim.agents.action_responder: 

Iteration 0: Base program full valset score: 0.12569444444444444 over 24 / 24 examples
Iteration 1: Selected program 0 score: 0.12569444444444444


2026-03-29 23:17:16 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.10
2026-03-29 23:17:16 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=True, completion_score=0.80
23:17:16 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:17:16 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
23:17:16 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:17:16 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:17:16 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.20
23:17:16 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:17:16 [INFO] LiteLLM: 
LiteLLM completion() model= 

Iteration 1: Proposed new text for action-responder-system: You are an expert Wilderness First Responder (WFR) instructor guiding students through realistic wilderness emergency scenarios.
Your goal is to help students learn proper assessment and treatment protocols while maintaining an engaging, supportive learning environment.

=== YOUR TASK ===
Evaluate the student's latest action within a multi-turn simulation. Your response must contain three outputs:
1.  **`was_correct`** (boolean): Determine if the student's action was medically correct.
    *   Set to `false` ONLY if the student attempts a treatment action (e.g., splint, bandage, medication) WITHOUT having performed the necessary prior assessment.
    *   Set to `true` for all other actions, including assessment steps, "I don't know" responses, and correct treatments following proper assessment. Assessment is ALWAYS considered correct.
    *   Important: The student action `"idk"` (I don't know) is a valid input indicating unce

2026-03-29 23:19:39 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.10
2026-03-29 23:19:39 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.40
23:19:39 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:19:39 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
23:19:39 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:19:39 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:19:40 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=True, completion_score=0.80
23:19:40 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:19:40 [INFO] LiteLLM: 
LiteLLM completion() model= 

...

2026-03-29 23:35:26 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.00
2026-03-29 23:35:27 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.40
23:35:27 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:35:27 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:35:27 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=True, completion_score=0.40
2026-03-29 23:35:27 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.06
23:35:27 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:35:27 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:35:27 [INFO] summit_sim.agents.action_respond

Iteration 4: Valset score for new program: 0.19791666666666666 (coverage 24 / 24)
Iteration 4: Val aggregate for new program: 0.19791666666666666
Iteration 4: Individual valset scores for new program: {0: 0.0, 1: 0.26666666666666666, 2: 0.0, 3: 0.3, 4: 0.5333333333333333, 5: 0.0, 6: 0.5, 7: 0.0, 8: 0.39999999999999997, 9: 0.3333333333333333, 10: 0.0, 11: 0.0, 12: 0.19999999999999998, 13: 0.3333333333333333, 14: 0.03333333333333333, 15: 0.3333333333333333, 16: 0.0, 17: 0.6666666666666666, 18: 0.0, 19: 0.0, 20: 0.3333333333333333, 21: 0.0, 22: 0.43333333333333335, 23: 0.08333333333333333}
Iteration 4: Objective aggregate scores for new program: {'completion-judge': 0.24375, 'quality-judge': 0.19166666666666665, 'medical-judge': 0.15833333333333333}
Iteration 4: New valset pareto front scores: {0: 0.3333333333333333, 1: 0.26666666666666666, 2: 0.19999999999999998, 3: 0.6, 4: 0.5333333333333333, 5: 0.06666666666666667, 6: 0.5, 7: 0.3333333333333333, 8: 0.39999999999999997, 9: 0.66666666666



2026-03-29 23:37:25 [INFO] summit_sim.agents.action_responder: Processing student action: turn=1/5, action_length=56
2026-03-29 23:37:25 [INFO] summit_sim.agents.action_responder: Processing student action: turn=2/5, action_length=3
2026-03-29 23:37:25 [INFO] summit_sim.agents.action_responder: Processing student action: turn=1/5, action_length=3
2026-03-29 23:37:25 [INFO] openai._base_client: Retrying request to /chat/completions in 0.380185 seconds


Iteration 5: Selected program 2 score: 0.19791666666666666


2026-03-29 23:37:26 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.20
2026-03-29 23:37:26 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=False, completion_score=0.05
23:37:27 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:37:27 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
23:37:27 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:37:27 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:37:28 [INFO] summit_sim.agents.action_responder: Action processed: was_correct=True, completion_score=0.70
23:37:28 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= deepseek/deepseek-v3.2; provider = openrouter
2026-03-29 23:37:28 [INFO] LiteLLM: 
LiteLLM completion() model= 

In [38]:
optimized_system_prompt = result.optimized_prompts[0]
optimized_user_prompt = result.optimized_prompts[1]

print(f"Optimized system prompt URI: {optimized_system_prompt.uri}")
print(f"Optimized system template: {optimized_system_prompt.template}")
print(f"Optimized user prompt URI: {optimized_user_prompt.uri}")
print(f"Optimized user template: {optimized_user_prompt.template}")

NameError: name 'result' is not defined

In [ ]:
models_to_test = [
    "google/gemini-3.1-flash-lite-preview",
    "qwen/qwen3.5-flash-02-23",
    "qwen/qwen3-235b-a22b-2507",
    "openai/gpt-4o-mini",
    "openai/gpt-4.1-mini",
    "openai/gpt-5-nano",
    "openai/gpt-5.4-nano",
    "openai/gpt-oss-120b",
    "deepseek/deepseek-v3.2",
    "x-ai/grok-4.1-fast",
    "xiaomi/mimo-v2-flash",
    "z-ai/glm-4.7-flash",
    "",
]